In [2]:
# Load Data

import pandas as pd

customers = pd.read_csv("customers_202604021940.csv")
orders = pd.read_csv("orders_202604021932.csv")
order_lines = pd.read_csv("order_lines_202604021931.csv")
products = pd.read_csv("products_202604021932.csv")

In [4]:
# Initial Exploration

# Preview data
print(orders.head())

# Check structure
print(orders.info())

# Check missing values
print(orders.isnull().sum())

   OrderID   OrderDate  CustomerID    Channel     Region PaymentMethod  \
0        1  2024-06-20         192  Wholesale  Northeast          Card   
1        2  2024-01-27          69   In-Store    Midwest           ACH   
2        3  2024-09-24          68     Online    Midwest        PayPal   
3        4  2024-02-13         130     Online      South          Card   
4        5  2025-01-15          52  Wholesale       West          Card   

  OrderStatus  
0    Refunded  
1   Completed  
2    Refunded  
3   Completed  
4   Completed  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   OrderID        700 non-null    int64 
 1   OrderDate      700 non-null    object
 2   CustomerID     700 non-null    int64 
 3   Channel        700 non-null    object
 4   Region         700 non-null    object
 5   PaymentMethod  700 non-null    object
 6   OrderS

In [5]:
# Data Cleaning

# Remove duplicates
orders = orders.drop_duplicates()
order_lines = order_lines.drop_duplicates()

# Handle missing values
orders = orders.dropna()

# Convert date columns
orders['OrderDate'] = pd.to_datetime(orders['OrderDate'])

# Standardized column names
orders.columns = orders.columns.str.strip().str.lower()

In [7]:
# clean all column names
for df_name in [order_lines, orders, products, customers]:
    df_name.columns = df_name.columns.str.strip().str.lower()

In [8]:
# Merge datasets

df = order_lines.merge(orders, on='orderid') \
                .merge(products, on='productid') \
                .merge(customers, on='customerid')

In [10]:
print(df.columns)

Index(['orderlineid', 'orderid', 'productid', 'quantity', 'unitprice',
       'discountpct', 'revenue', 'orderdate', 'customerid', 'channel',
       'region_x', 'paymentmethod', 'orderstatus', 'productname', 'category',
       'baseprice', 'issubscription', 'firstname', 'lastname', 'email',
       'segment', 'region_y', 'state', 'signupdate'],
      dtype='object')


In [11]:
df['calculated_revenue'] = df['quantity'] * df['unitprice']

In [12]:
# Create Revenue Column
df['revenue']= df['quantity'] * df['unitprice']

In [13]:
# Revenue Trends Over Time
df['month'] = df['orderdate'].dt.to_period('M')
monthly_revenue = df.groupby('month')['revenue'].sum().sort_values(ascending=False)
print(monthly_revenue)


month
2024-01    34210.46
2024-02    27444.56
2024-12    26688.39
2024-06    26425.31
2025-02    25034.43
2024-07    24648.54
2024-03    22945.89
2024-04    20992.57
2025-01    20682.12
2024-11    16109.83
2024-10    15970.36
2024-05    15382.21
2024-08    15019.66
2024-09    14513.45
Freq: M, Name: revenue, dtype: float64


In [14]:
# Category Performance
category_revenue = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print(category_revenue)

category
Home Office         77549.57
Wellness            76156.71
Tech Accessories    74076.37
Dental Supplies     45397.05
Personal Care       32888.08
Name: revenue, dtype: float64


In [15]:
# Customer Dependency
top_customers = df.groupby('customerid')['revenue'].sum().sort_values(ascending=False)
print(top_customers)

customerid
80     6005.72
40     5445.85
165    5131.52
16     4941.50
60     4876.35
        ...   
194      80.16
105      76.88
96       60.85
100      53.69
30       44.02
Name: revenue, Length: 190, dtype: float64


In [16]:
# Average Order Value (AOV)
aov = df.groupby('orderid')['revenue'].sum().mean()
print(aov)

581.8779087452472


In [18]:
df = df.rename(columns={
     'region_x': 'customer_region',
     'region_y': 'order_region'
})

In [19]:
region_revenue = df.groupby('customer_region')['revenue'].sum().sort_values(ascending=False)
print(region_revenue)

customer_region
Northeast    95181.78
Midwest      72046.70
South        71919.82
West         66919.48
Name: revenue, dtype: float64
